# 📈 Étape 1 — Données de marché (multi-sources)
### News-Driven Regime Attribution Engine
---
Ce notebook couvre l'**étape 1** du pipeline : récupération et préparation des données de marché (NASDAQ + VIX) via `yfinance`, calcul des variables dérivées utiles à la détection de régime.

**Contenu :**
1. Installation & imports
2. Récupération des données NASDAQ (`^IXIC`)
3. Calcul des variables dérivées (Returns, Volatility, RSI, MACD, Drawdown)
4. Ajout du VIX (`^VIX`)
5. Vérification qualité & statistiques descriptives
6. Visualisations exploratoires
7. Export du dataset


## 1. Installation des dépendances

In [13]:
# Installation des dépendances (à exécuter une seule fois)
!pip install pandas-datareader yfinance pandas numpy matplotlib seaborn plotly requests

# Vérification des packages disponibles
import importlib, sys
packages = ["pandas_datareader", "yfinance", "pandas", "numpy", "matplotlib", "seaborn", "requests"]
for pkg in packages:
    status = "✅" if importlib.util.find_spec(pkg.replace("-","_")) else "❌"
    print(f"  {status}  {pkg}")

  ✅  pandas_datareader
  ✅  yfinance
  ✅  pandas
  ✅  numpy
  ✅  matplotlib
  ✅  seaborn
  ✅  requests



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import requests
import io
from datetime import datetime, timedelta

# Imports optionnels (avec fallback propre)
try:
    import pandas_datareader.data as web
    HAS_DATAREADER = True
except ImportError as e:
    HAS_DATAREADER = False
    print(f"⚠️  pandas-datareader non disponible — Erreur: {e}")

try:
    import yfinance as yf
    HAS_YFINANCE = True
except ImportError:
    HAS_YFINANCE = False
    print("⚠️  yfinance non disponible — pip install yfinance")

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 110

print(f"pandas-datareader : {'✅' if HAS_DATAREADER else '❌'}")
print(f"yfinance          : {'✅' if HAS_YFINANCE else '❌'}")
print("✅ Imports OK")

TypeError: deprecate_kwarg() missing 1 required positional argument: 'new_arg_name'

# 📈 Étape 1 — Données de marché (multi-sources)
### News-Driven Regime Attribution Engine
---
Ce notebook couvre l'**étape 1** du pipeline : récupération des données de marché NASDAQ + VIX.

Il propose **3 méthodes de téléchargement** avec fallback automatique :
1. **`pandas-datareader` + Stooq** — gratuit, sans clé API ✅ *(méthode principale)*
2. **`yfinance`** — fallback si Stooq est indisponible
3. **`Alpha Vantage`** — fallback avec clé API gratuite (500 req/jour)

**Contenu :**
1. Installation & imports
2. Fonctions de téléchargement (multi-sources avec fallback)
3. Paramètres globaux
4. Récupération NASDAQ + VIX
5. Variables dérivées : Returns, Volatility, RSI, MACD, Drawdown
6. Vérification qualité & statistiques
7. Visualisations exploratoires
8. Export du dataset


## 1. Installation des dépendances

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
# !pip install pandas-datareader yfinance pandas numpy matplotlib seaborn plotly requests

# Vérification des packages disponibles
import importlib, sys
packages = ["pandas_datareader", "yfinance", "pandas", "numpy", "matplotlib", "seaborn", "requests"]
for pkg in packages:
    status = "✅" if importlib.util.find_spec(pkg.replace("-","_")) else "❌"
    print(f"  {status}  {pkg}")


## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
import requests
import io
from datetime import datetime, timedelta

# Imports optionnels (avec fallback propre)
try:
    import pandas_datareader.data as web
    HAS_DATAREADER = True
except ImportError:
    HAS_DATAREADER = False
    print("⚠️  pandas-datareader non disponible — pip install pandas-datareader==0.10.0")

try:
    import yfinance as yf
    HAS_YFINANCE = True
except ImportError:
    HAS_YFINANCE = False
    print("⚠️  yfinance non disponible — pip install yfinance")

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 110

print(f"pandas-datareader : {'✅' if HAS_DATAREADER else '❌'}")
print(f"yfinance          : {'✅' if HAS_YFINANCE else '❌'}")
print("✅ Imports OK")


## 3. Paramètres globaux

In [ ]:
# ── Paramètres ──────────────────────────────────────────────────────────────
TICKER_NASDAQ   = "^IXIC"   # NASDAQ Composite  (Stooq: ^IXIC | yfinance: ^IXIC)
TICKER_NASDAQ_STOOQ = "^ixic"  # Stooq utilise des minuscules pour les indices
TICKER_VIX      = "^VIX"
TICKER_VIX_STOOQ = "^vix"
START_DATE      = "2014-01-01"
END_DATE        = "2024-12-31"
ROLLING_VOL     = 20
ROLLING_RSI     = 14
MACD_FAST       = 12
MACD_SLOW       = 26
MACD_SIGNAL     = 9
MAX_DD_WINDOW   = 252

# Clé Alpha Vantage (optionnelle — gratuite sur https://www.alphavantage.co/support/#api-key)
ALPHA_VANTAGE_KEY = "YOUR_KEY_HERE"

print(f"Période : {START_DATE} → {END_DATE}")


## 4. Récupération des données NASDAQ + VIX

### Méthode principale : `pandas-datareader` + **Stooq**
Stooq (https://stooq.com) est une source fiable, **gratuite et sans clé API**, qui couvre les indices mondiaux avec un historique de plusieurs décennies. C'est la source recommandée en remplacement de yfinance.

> **Tickers Stooq** : les indices utilisent des minuscules — `^ixic` (NASDAQ), `^vix` (VIX), `^spx` (S&P 500).

### Méthode 2 : `yfinance` (fallback automatique)
Si Stooq est indisponible, le code bascule sur yfinance.

### Méthode 3 : `Alpha Vantage` (fallback avec clé API)
Clé gratuite sur [alphavantage.co](https://www.alphavantage.co/support/#api-key) — 500 req/jour, 15 req/min.


In [ ]:
def fetch_stooq(ticker_stooq: str, start: str, end: str) -> pd.DataFrame:
    """Télécharge OHLCV depuis Stooq via pandas-datareader (sans clé API)."""
    df = web.DataReader(ticker_stooq, "stooq", start=start, end=end)
    df = df.sort_index()                         # Stooq retourne en ordre décroissant
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "date"
    return df


def fetch_yfinance(ticker: str, start: str, end: str) -> pd.DataFrame:
    """Télécharge via yfinance (fallback)."""
    t = yf.Ticker(ticker)
    df = t.history(start=start, end=end)
    df = df[["Open", "High", "Low", "Close", "Volume"]]
    df.index = pd.to_datetime(df.index).tz_localize(None)
    df.index.name = "date"
    return df


def fetch_alpha_vantage(ticker: str, api_key: str) -> pd.DataFrame:
    """
    Télécharge l'historique complet (20 ans) via Alpha Vantage.
    ticker : symbole US — ex. 'QQQ' pour le NASDAQ-100 ETF, 'SPY' pour S&P 500.
    Clé gratuite : https://www.alphavantage.co/support/#api-key
    """
    url = (
        "https://www.alphavantage.co/query"
        f"?function=TIME_SERIES_DAILY_ADJUSTED"
        f"&symbol={ticker}&outputsize=full&datatype=json&apikey={api_key}"
    )
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json().get("Time Series (Daily)", {})
    df = pd.DataFrame(data).T
    df.index = pd.to_datetime(df.index)
    df.index.name = "date"
    df = df.rename(columns={
        "1. open": "Open", "2. high": "High", "3. low": "Low",
        "4. close": "Close", "6. volume": "Volume"
    })[["Open", "High", "Low", "Close", "Volume"]].astype(float).sort_index()
    return df


def load_market_data(ticker_stooq: str, ticker_yf: str,
                     start: str, end: str,
                     av_ticker: str = None, av_key: str = None) -> pd.DataFrame:
    """
    Essaie les sources dans l'ordre : Stooq → yfinance → Alpha Vantage.
    Retourne un DataFrame avec colonnes [Open, High, Low, Close, Volume].
    """
    # ── Méthode 1 : Stooq ──────────────────────────────────────────────────
    if HAS_DATAREADER:
        try:
            df = fetch_stooq(ticker_stooq, start, end)
            print(f"✅ Stooq  : {len(df)} lignes chargées ({df.index.min().date()} → {df.index.max().date()})")
            return df
        except Exception as e:
            print(f"⚠️  Stooq échoué : {e}")

    # ── Méthode 2 : yfinance ───────────────────────────────────────────────
    if HAS_YFINANCE:
        try:
            df = fetch_yfinance(ticker_yf, start, end)
            print(f"✅ yfinance : {len(df)} lignes chargées ({df.index.min().date()} → {df.index.max().date()})")
            return df
        except Exception as e:
            print(f"⚠️  yfinance échoué : {e}")

    # ── Méthode 3 : Alpha Vantage ──────────────────────────────────────────
    if av_ticker and av_key and av_key != "YOUR_KEY_HERE":
        try:
            df = fetch_alpha_vantage(av_ticker, av_key)
            df = df.loc[start:end]
            print(f"✅ Alpha Vantage : {len(df)} lignes chargées")
            return df
        except Exception as e:
            print(f"⚠️  Alpha Vantage échoué : {e}")

    raise RuntimeError(
        "❌ Toutes les sources ont échoué.\n"
        "  → Vérifiez votre connexion internet\n"
        "  → Ou fournissez une clé Alpha Vantage dans ALPHA_VANTAGE_KEY"
    )


# ── Chargement NASDAQ ──────────────────────────────────────────────────────
print("── Chargement NASDAQ ──")
df_raw = load_market_data(
    ticker_stooq=TICKER_NASDAQ_STOOQ,
    ticker_yf=TICKER_NASDAQ,
    start=START_DATE,
    end=END_DATE,
    av_ticker="QQQ",          # ETF NASDAQ-100, proxy pour Alpha Vantage
    av_key=ALPHA_VANTAGE_KEY
)

df = df_raw[["Close", "Volume"]].copy()

print(f"\nShape : {df.shape}")
df.tail(3)


## 5. Calcul des variables dérivées

### 5.1 Rendements & Volatilité glissante


In [ ]:
# Rendement journalier
df["Returns"] = df["Close"].pct_change()

# Volatilité glissante (écart-type des rendements sur 20 jours)
df["Volatility"] = df["Returns"].rolling(ROLLING_VOL).std()

df.dropna(subset=["Returns", "Volatility"], inplace=True)

print(f"Shape après dropna : {df.shape}")
df[["Close", "Returns", "Volatility"]].describe().round(4)


### 5.2 RSI (Relative Strength Index)

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """RSI classique (méthode Wilder)."""
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI"] = compute_rsi(df["Close"], period=ROLLING_RSI)

print("RSI calculé ✅")
print(df["RSI"].describe().round(2))


### 5.3 MACD (Moving Average Convergence/Divergence)

In [ ]:
def compute_macd(series: pd.Series,
                 fast: int = 12, slow: int = 26, signal: int = 9):
    """Retourne MACD line, signal line et histogramme."""
    ema_fast   = series.ewm(span=fast,   adjust=False).mean()
    ema_slow   = series.ewm(span=slow,   adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_line
    return macd_line, signal_line, histogram

df["MACD"], df["MACD_signal"], df["MACD_hist"] = compute_macd(
    df["Close"], MACD_FAST, MACD_SLOW, MACD_SIGNAL
)

print("MACD calculé ✅")
df[["MACD", "MACD_signal", "MACD_hist"]].tail(3)


### 5.4 Drawdown maximum glissant

In [ ]:
def compute_rolling_max_drawdown(close: pd.Series, window: int = 252) -> pd.Series:
    """Drawdown max glissant sur `window` jours."""
    rolling_max = close.rolling(window, min_periods=1).max()
    drawdown    = (close - rolling_max) / rolling_max
    return drawdown

df["Drawdown"] = compute_rolling_max_drawdown(df["Close"], MAX_DD_WINDOW)

print("Drawdown calculé ✅")
print(f"Drawdown min (pire correction) : {df['Drawdown'].min():.2%}")
print(f"Drawdown moyen                 : {df['Drawdown'].mean():.2%}")


## 6. Ajout du VIX (`^VIX`)
Le VIX mesure la volatilité implicite du S&P 500. C'est un excellent proxy des périodes de stress de marché.


In [ ]:
print("── Chargement VIX ──")
vix_raw = load_market_data(
    ticker_stooq=TICKER_VIX_STOOQ,
    ticker_yf=TICKER_VIX,
    start=START_DATE,
    end=END_DATE,
    av_ticker=None,   # VIX non disponible sur Alpha Vantage free tier
    av_key=None
)

vix_series = vix_raw["Close"]
vix_series.index.name = "date"

# Réindexer sur le calendrier NASDAQ + forward-fill les jours manquants
df["VIX"] = vix_series.reindex(df.index).ffill()

print(f"VIX ajouté ✅  —  valeurs manquantes : {df['VIX'].isnull().sum()}")
print(df["VIX"].describe().round(2))


## 7. Vérification qualité & statistiques descriptives

In [ ]:
print("=" * 50)
print(f"Shape final      : {df.shape}")
print(f"Période          : {df.index.min().date()} → {df.index.max().date()}")
print(f"Colonnes         : {list(df.columns)}")
print(f"Valeurs nulles   :")
print(df.isnull().sum())
print("=" * 50)
df.describe().round(4)


In [ ]:
# Affichage des 5 premières et 5 dernières lignes
print("── HEAD ──")
display(df.head())
print("── TAIL ──")
display(df.tail())


## 8. Visualisations exploratoires

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 22), sharex=True)
fig.suptitle("NASDAQ (^IXIC) — Vue d'ensemble 2014-2024", fontsize=15, fontweight="bold", y=1.01)

# ── Prix ──
axes[0].plot(df.index, df["Close"], color="#2196F3", linewidth=1.2)
axes[0].set_ylabel("Prix clôture ($)")
axes[0].set_title("Prix de clôture")

# ── Rendements ──
axes[1].bar(df.index, df["Returns"], color=np.where(df["Returns"] >= 0, "#4CAF50", "#F44336"),
            width=1, alpha=0.7)
axes[1].axhline(0, color="white", linewidth=0.5)
axes[1].set_ylabel("Rendement journalier")
axes[1].set_title("Rendements journaliers")

# ── Volatilité glissante ──
axes[2].fill_between(df.index, df["Volatility"], alpha=0.6, color="#FF9800")
axes[2].set_ylabel("Volatilité (σ)")
axes[2].set_title(f"Volatilité glissante ({ROLLING_VOL}j)")

# ── VIX ──
axes[3].fill_between(df.index, df["VIX"], alpha=0.5, color="#9C27B0")
axes[3].axhline(20, color="white", linestyle="--", linewidth=0.8, label="Seuil 20 (attention)")
axes[3].axhline(30, color="#F44336", linestyle="--", linewidth=0.8, label="Seuil 30 (stress)")
axes[3].legend(fontsize=8)
axes[3].set_ylabel("VIX")
axes[3].set_title("VIX — Indice de volatilité implicite")

# ── Drawdown ──
axes[4].fill_between(df.index, df["Drawdown"] * 100, 0, alpha=0.6, color="#F44336")
axes[4].set_ylabel("Drawdown (%)")
axes[4].set_title(f"Drawdown max glissant ({MAX_DD_WINDOW}j)")
axes[4].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

for ax in axes:
    ax.margins(x=0.01)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/nasdaq_overview.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure sauvegardée ✅")


In [ ]:
# ── RSI ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df["RSI"], color="#00BCD4", linewidth=1)
ax.axhline(70, color="#F44336", linestyle="--", label="Surachat (70)")
ax.axhline(30, color="#4CAF50", linestyle="--", label="Survente (30)")
ax.fill_between(df.index, df["RSI"], 70, where=(df["RSI"] > 70), alpha=0.3, color="#F44336")
ax.fill_between(df.index, df["RSI"], 30, where=(df["RSI"] < 30), alpha=0.3, color="#4CAF50")
ax.set_ylim(0, 100)
ax.set_ylabel("RSI")
ax.set_title(f"RSI ({ROLLING_RSI}j) — NASDAQ")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


In [ ]:
# ── MACD ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(df.index, df["Close"], color="#2196F3", linewidth=1)
ax1.set_ylabel("Prix ($)")
ax1.set_title("NASDAQ — Prix & MACD")

ax2.plot(df.index, df["MACD"],        color="#4CAF50", linewidth=1, label="MACD")
ax2.plot(df.index, df["MACD_signal"], color="#F44336", linewidth=1, label="Signal")
ax2.bar(df.index, df["MACD_hist"],
        color=np.where(df["MACD_hist"] >= 0, "#4CAF50", "#F44336"),
        alpha=0.4, width=1)
ax2.axhline(0, color="white", linewidth=0.5)
ax2.set_ylabel("MACD")
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()


In [ ]:
# ── Matrice de corrélation ────────────────────────────────────────────────────
cols = ["Returns", "Volatility", "RSI", "MACD", "Drawdown", "VIX"]
corr_matrix = df[cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, mask=mask, ax=ax,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Matrice de corrélation — variables dérivées")
plt.tight_layout()
plt.show()


## 9. Export du dataset

In [ ]:
import os
os.makedirs("data/market", exist_ok=True)

output_path = "data/market/nasdaq_features.csv"
df.to_csv(output_path)

print(f"✅ Dataset exporté → {output_path}")
print(f"   Shape   : {df.shape}")
print(f"   Colonnes: {list(df.columns)}")


## ✅ Résumé de l'étape 1

| Variable | Description | Utilité pour la détection de régime |
|---|---|---|
| `Close` | Prix de clôture journalier | Référence de prix |
| `Volume` | Volume échangé | Liquidité, conviction |
| `Returns` | Rendement journalier (%) | Signal brut |
| `Volatility` | Écart-type glissant 20j | Proxy du risque |
| `RSI` | Indice de force relative 14j | Surachat / survente |
| `MACD` | Convergence/Divergence des MM | Momentum |
| `Drawdown` | Recul max glissant 252j | Amplitude des corrections |
| `VIX` | Volatilité implicite S&P | Stress systémique |

**Prochaine étape → Étape 2 : Détection de régime (HMM / Bayesian Change Point)**


## 3. Paramètres globaux

In [3]:
# ── Paramètres ──────────────────────────────────────────────────────────────
TICKER_NASDAQ = "^IXIC"   # NASDAQ Composite
TICKER_VIX    = "^VIX"    # CBOE Volatility Index
START_DATE    = "2014-01-01"
END_DATE      = "2024-12-31"
ROLLING_VOL   = 20          # fenêtre volatilité (jours ouvrés)
ROLLING_RSI   = 14          # fenêtre RSI standard
MACD_FAST     = 12
MACD_SLOW     = 26
MACD_SIGNAL   = 9
MAX_DD_WINDOW = 252         # fenêtre drawdown max (~1 an)

print(f"Période : {START_DATE} → {END_DATE}")
print(f"Tickers  : {TICKER_NASDAQ}, {TICKER_VIX}")


Période : 2014-01-01 → 2024-12-31
Tickers  : ^IXIC, ^VIX


## 4. Récupération des données NASDAQ (`^IXIC`)
On télécharge 10 ans de données journalières depuis Yahoo Finance via `yfinance` avec `yfinance.download()`.

In [4]:
import pandas as pd
import requests
from io import StringIO
from datetime import datetime

print("Téléchargement NASDAQ via Yahoo Finance (requests)...")

# Convertir dates en timestamps Unix
start_ts = int(datetime.strptime(START_DATE, "%Y-%m-%d").timestamp())
end_ts = int(datetime.strptime(END_DATE, "%Y-%m-%d").timestamp())

url = f"https://query1.finance.yahoo.com/v7/finance/download/^IXIC?period1={start_ts}&period2={end_ts}&interval=1d&events=history"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

try:
    response = requests.get(url, headers=headers, timeout=15, verify=False)
    response.raise_for_status()
    
    df_raw = pd.read_csv(StringIO(response.text))
    df_raw['Date'] = pd.to_datetime(df_raw['Date'])
    df_raw.set_index('Date', inplace=True)
    print("✅ Téléchargement réussi")
    
except Exception as e:
    print(f"❌ Erreur : {e}")
    raise

df = df_raw[["Close", "Volume"]].copy()
df.index.name = "date"
df = df.dropna(subset=['Close'])

print(f"Shape brute      : {df.shape}")
print(f"Période couverte : {df.index.min().date()} → {df.index.max().date()}")
print(f"Valeurs manquantes :\n{df.isnull().sum()}")
df.tail(5)

Téléchargement NASDAQ via Yahoo Finance (requests)...
❌ Erreur : 401 Client Error: Unauthorized for url: https://query1.finance.yahoo.com/v7/finance/download/%5EIXIC?period1=1388530800&period2=1735599600&interval=1d&events=history


HTTPError: 401 Client Error: Unauthorized for url: https://query1.finance.yahoo.com/v7/finance/download/%5EIXIC?period1=1388530800&period2=1735599600&interval=1d&events=history

## 5. Calcul des variables dérivées

### 5.1 Rendements & Volatilité glissante


In [ ]:
# Rendement journalier
df["Returns"] = df["Close"].pct_change()

# Volatilité glissante (écart-type des rendements sur 20 jours)
df["Volatility"] = df["Returns"].rolling(ROLLING_VOL).std()

df.dropna(subset=["Returns", "Volatility"], inplace=True)

print(f"Shape après dropna : {df.shape}")
df[["Close", "Returns", "Volatility"]].describe().round(4)


### 5.2 RSI (Relative Strength Index)

In [ ]:
def compute_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    """RSI classique (méthode Wilder)."""
    delta = series.diff()
    gain  = delta.clip(lower=0)
    loss  = -delta.clip(upper=0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs  = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI"] = compute_rsi(df["Close"], period=ROLLING_RSI)

print("RSI calculé ✅")
print(df["RSI"].describe().round(2))


### 5.3 MACD (Moving Average Convergence/Divergence)

In [ ]:
def compute_macd(series: pd.Series,
                 fast: int = 12, slow: int = 26, signal: int = 9):
    """Retourne MACD line, signal line et histogramme."""
    ema_fast   = series.ewm(span=fast,   adjust=False).mean()
    ema_slow   = series.ewm(span=slow,   adjust=False).mean()
    macd_line  = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram  = macd_line - signal_line
    return macd_line, signal_line, histogram

df["MACD"], df["MACD_signal"], df["MACD_hist"] = compute_macd(
    df["Close"], MACD_FAST, MACD_SLOW, MACD_SIGNAL
)

print("MACD calculé ✅")
df[["MACD", "MACD_signal", "MACD_hist"]].tail(3)


### 5.4 Drawdown maximum glissant

In [ ]:
def compute_rolling_max_drawdown(close: pd.Series, window: int = 252) -> pd.Series:
    """Drawdown max glissant sur `window` jours."""
    rolling_max = close.rolling(window, min_periods=1).max()
    drawdown    = (close - rolling_max) / rolling_max
    return drawdown

df["Drawdown"] = compute_rolling_max_drawdown(df["Close"], MAX_DD_WINDOW)

print("Drawdown calculé ✅")
print(f"Drawdown min (pire correction) : {df['Drawdown'].min():.2%}")
print(f"Drawdown moyen                 : {df['Drawdown'].mean():.2%}")


## 6. Ajout du VIX (`^VIX`)
Le VIX mesure la volatilité implicite du S&P 500. C'est un excellent proxy des périodes de stress de marché.

In [ ]:
print("Téléchargement VIX en cours...")
vix_raw = yf.download(TICKER_VIX, start=START_DATE, end=END_DATE, progress=False)["Close"]
vix_raw.index = pd.to_datetime(vix_raw.index).tz_localize(None)
vix_raw.index.name = "date"

# Réindexer sur le calendrier NASDAQ + forward-fill les jours manquants
df["VIX"] = vix_raw.reindex(df.index).ffill()

print(f"VIX ajouté ✅  —  valeurs manquantes : {df['VIX'].isnull().sum()}")
print(df["VIX"].describe().round(2))

## 7. Vérification qualité & statistiques descriptives

In [ ]:
print("=" * 50)
print(f"Shape final      : {df.shape}")
print(f"Période          : {df.index.min().date()} → {df.index.max().date()}")
print(f"Colonnes         : {list(df.columns)}")
print(f"Valeurs nulles   :")
print(df.isnull().sum())
print("=" * 50)
df.describe().round(4)


In [ ]:
# Affichage des 5 premières et 5 dernières lignes
print("── HEAD ──")
display(df.head())
print("── TAIL ──")
display(df.tail())


## 8. Visualisations exploratoires

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 22), sharex=True)
fig.suptitle("NASDAQ (^IXIC) — Vue d'ensemble 2014-2024", fontsize=15, fontweight="bold", y=1.01)

# ── Prix ──
axes[0].plot(df.index, df["Close"], color="#2196F3", linewidth=1.2)
axes[0].set_ylabel("Prix clôture ($)")
axes[0].set_title("Prix de clôture")

# ── Rendements ──
axes[1].bar(df.index, df["Returns"], color=np.where(df["Returns"] >= 0, "#4CAF50", "#F44336"),
            width=1, alpha=0.7)
axes[1].axhline(0, color="white", linewidth=0.5)
axes[1].set_ylabel("Rendement journalier")
axes[1].set_title("Rendements journaliers")

# ── Volatilité glissante ──
axes[2].fill_between(df.index, df["Volatility"], alpha=0.6, color="#FF9800")
axes[2].set_ylabel("Volatilité (σ)")
axes[2].set_title(f"Volatilité glissante ({ROLLING_VOL}j)")

# ── VIX ──
axes[3].fill_between(df.index, df["VIX"], alpha=0.5, color="#9C27B0")
axes[3].axhline(20, color="white", linestyle="--", linewidth=0.8, label="Seuil 20 (attention)")
axes[3].axhline(30, color="#F44336", linestyle="--", linewidth=0.8, label="Seuil 30 (stress)")
axes[3].legend(fontsize=8)
axes[3].set_ylabel("VIX")
axes[3].set_title("VIX — Indice de volatilité implicite")

# ── Drawdown ──
axes[4].fill_between(df.index, df["Drawdown"] * 100, 0, alpha=0.6, color="#F44336")
axes[4].set_ylabel("Drawdown (%)")
axes[4].set_title(f"Drawdown max glissant ({MAX_DD_WINDOW}j)")
axes[4].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

for ax in axes:
    ax.margins(x=0.01)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/nasdaq_overview.png", dpi=130, bbox_inches="tight")
plt.show()
print("Figure sauvegardée ✅")


In [ ]:
# ── RSI ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.index, df["RSI"], color="#00BCD4", linewidth=1)
ax.axhline(70, color="#F44336", linestyle="--", label="Surachat (70)")
ax.axhline(30, color="#4CAF50", linestyle="--", label="Survente (30)")
ax.fill_between(df.index, df["RSI"], 70, where=(df["RSI"] > 70), alpha=0.3, color="#F44336")
ax.fill_between(df.index, df["RSI"], 30, where=(df["RSI"] < 30), alpha=0.3, color="#4CAF50")
ax.set_ylim(0, 100)
ax.set_ylabel("RSI")
ax.set_title(f"RSI ({ROLLING_RSI}j) — NASDAQ")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()


In [ ]:
# ── MACD ─────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(df.index, df["Close"], color="#2196F3", linewidth=1)
ax1.set_ylabel("Prix ($)")
ax1.set_title("NASDAQ — Prix & MACD")

ax2.plot(df.index, df["MACD"],        color="#4CAF50", linewidth=1, label="MACD")
ax2.plot(df.index, df["MACD_signal"], color="#F44336", linewidth=1, label="Signal")
ax2.bar(df.index, df["MACD_hist"],
        color=np.where(df["MACD_hist"] >= 0, "#4CAF50", "#F44336"),
        alpha=0.4, width=1)
ax2.axhline(0, color="white", linewidth=0.5)
ax2.set_ylabel("MACD")
ax2.legend()
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()


In [ ]:
# ── Matrice de corrélation ────────────────────────────────────────────────────
cols = ["Returns", "Volatility", "RSI", "MACD", "Drawdown", "VIX"]
corr_matrix = df[cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, mask=mask, ax=ax,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Matrice de corrélation — variables dérivées")
plt.tight_layout()
plt.show()


## 9. Export du dataset

In [ ]:
import os
os.makedirs("data/market", exist_ok=True)

output_path = "data/market/nasdaq_features.csv"
df.to_csv(output_path)

print(f"✅ Dataset exporté → {output_path}")
print(f"   Shape   : {df.shape}")
print(f"   Colonnes: {list(df.columns)}")


## ✅ Résumé de l'étape 1

| Variable | Description | Utilité pour la détection de régime |
|---|---|---|
| `Close` | Prix de clôture journalier | Référence de prix |
| `Volume` | Volume échangé | Liquidité, conviction |
| `Returns` | Rendement journalier (%) | Signal brut |
| `Volatility` | Écart-type glissant 20j | Proxy du risque |
| `RSI` | Indice de force relative 14j | Surachat / survente |
| `MACD` | Convergence/Divergence des MM | Momentum |
| `Drawdown` | Recul max glissant 252j | Amplitude des corrections |
| `VIX` | Volatilité implicite S&P | Stress systémique |

**Prochaine étape → Étape 2 : Détection de régime (HMM / Bayesian Change Point)**
